In [11]:
from collections import defaultdict

# first attempt
class Tokenizer:
    def __init__(self) -> None:
        self.merges: defaultdict[tuple[int, int], int] = defaultdict(int)
        self.count: defaultdict[tuple[int, int], int] = defaultdict(int)
        self.vocab = {x: bytes([x]) for x in range(256)}

    def encode(self, text: str):
        byted_text = text.encode("utf-8")
        indices = list(byted_text)
        merges = self.merge(indices, 3)
        return merges

    def merge(self, indices: list[int], merge_num):
        for i in range(merge_num):
            for idx, idx2 in zip(indices, indices[1:]):
                self.count[(idx, idx2)] = self.count[(idx, idx2)] + 1
            max_pair = max(self.count, key=lambda pos: self.count[pos])
            new_idx = 255 + i
            self.merges[max_pair] = new_idx
            self.vocab[new_idx] = self.vocab[max_pair[0]] + self.vocab[max_pair[1]]
            new_indices = []
            j = 0
            while j < len(indices):
                if (
                    j < len(indices) - 1
                    and indices[j] == max_pair[0]
                    and indices[j + 1] == max_pair[1]
                ):
                    new_indices.append(new_idx)
                    j += 2
                else:
                    new_indices.append(indices[j])
                    j += 1
            indices = new_indices
        return indices

    def decode(self, indices: list[int]):
        s = ""
        for idx in indices:
            byted_char = self.vocab[idx]
            s += bytes(byted_char).decode("utf-8")
        return s


tokenizer = Tokenizer()
encoded_text = tokenizer.encode("This is some text")
tokenizer.decode(encoded_text)


'This is some text'